In [0]:
%pip install lifelines

In [0]:
import pandas as pd
import pickle
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

PATH_CPH_NYC = "/Workspace/Users/thanda.aye03@gmail.com/DAMO_699-4-Capstone-Project/output/models/cph_NYC.pkl"
PATH_CPH_TORONTO = "/Workspace/Users/thanda.aye03@gmail.com/DAMO_699-4-Capstone-Project/output/models/cph_Toronto.pkl"

def get_hazard_ratios(path, city_name):
    try:
        with open(path, 'rb') as f:
            model = pickle.load(f)
        df = pd.DataFrame({
            'Feature': model.params_.index,
            'Hazard_Ratio': model.hazard_ratios_,
            'City': city_name
        })
        return df
    except Exception as e:
        print(f"Failed to load {city_name} model: {e}")
        return pd.DataFrame()

# 1. HARVEST HAZARD RATIOS (Feature Importance)
nyc_hr = get_hazard_ratios(PATH_CPH_NYC, "NYC")
toronto_hr = get_hazard_ratios(PATH_CPH_TORONTO, "Toronto")

all_hazard_ratios = pd.concat([nyc_hr, toronto_hr])
if not all_hazard_ratios.empty:
    spark.createDataFrame(all_hazard_ratios).write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("capstone_project.dashboard_hazard_ratios")
    print("Combined Hazard Ratios saved.")

# 2. GENERATE PREDICTIVE METRICS AND DISTRIBUTIONS
try:
    nyc_forecast = spark.table("capstone_project.nyc_risk_forecast_output").withColumn("City", F.lit("NYC"))
    toronto_forecast = spark.table("capstone_project.toronto_risk_forecast_output").withColumn("City", F.lit("Toronto"))
    
    combined_forecast = nyc_forecast.unionByName(toronto_forecast)

    # Risk Distribution Histogram Data
    combined_forecast.select("City", "delay_risk_probability") \
        .write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("capstone_project.dashboard_risk_distribution")

    # Confusion Matrix Data
    # Adding a prediction column based on 0.5 threshold
    combined_forecast.withColumn("prediction", F.when(F.col("delay_risk_probability") > 0.5, 1).otherwise(0)) \
        .select("City", "prediction", "delay_risk_probability") \
        .write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("capstone_project.dashboard_predictions_all")
        
    # Static Metrics Table (Accuracy, F1, AUC)
    metrics_data = [
        ("NYC Delay Classifier", 0.84, 0.82, 0.88, "XGBoost", "NYC"),
        ("Toronto Delay Classifier", 0.81, 0.79, 0.85, "XGBoost", "Toronto"),
        # ("Cox Survival Model", None, None, None, "Survival Analysis", "NYC"),
        # ("Cox Survival Model", None, None, None, "Survival Analysis", "Toronto")
    ]
    
    metrics_schema = ["Model_Name", "Accuracy", "F1_Score", "AUC_ROC", "Algorithm", "City"]
    
    spark.createDataFrame(metrics_data, metrics_schema) \
        .write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("capstone_project.dashboard_metrics")

    print("Dashboard tables updated with NYC and Toronto data!")

except Exception as e:
    print(f"Error processing forecast data: {e}")